# Empirical Strategy-Proofness Audit for LLM-Mediated RSD


This notebook audits the empirical manipulability of the language-report-to-allocation pipeline.

For each target student, the experiment replaces the truthful report with 100 structured fake personas expressed as natural-language reports. The preference model, other students' reports, course capacities, PE-calibrated weak threshold, and common RSD priority orders are held fixed. Outcomes are evaluated using the target student's hidden strict true preference.

Two mechanisms are compared:

1. **Strict LLM-proxy RSD**.
2. **PE-calibrated weak LLM-proxy RSD followed by proxy-Pareto completion**.

The audit reports ex-post profitable deviation rates, ex-ante expected-rank gains, empirical stochastic-dominance improvements, downside risk, best-of-K search curves, and success patterns by fake-report strategy. A profitable deviation is a counterexample to strategy-proofness; its observed frequency is conditional on the explicitly defined attack generator and is not an unconditional manipulation probability.


## 1. Install packages


In [ ]:
!python -m pip install -q -U "transformers>=4.56,<5" "accelerate>=1.2" "bitsandbytes>=0.46.1" scikit-learn scipy sentencepiece gdown

In [ ]:
import ast
import json
import random
import re
import shutil
import time
from itertools import combinations
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.optimize import Bounds, LinearConstraint, milp
from scipy.sparse import lil_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm

SEED = 20260909
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
N_FOLDS = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


## 2. Load the synthetic course-allocation data


In [ ]:
from pathlib import Path
import json
import os
import shutil
import zipfile

import gdown
import numpy as np
import pandas as pd

# Use a local dataset ZIP, or opt in to a link-shared Drive download.
ZIP_PATH = Path("fixed_tolerance_course_data.zip")
FILE_ID = os.environ.get("COURSE_DATA_DRIVE_FILE_ID", "").strip()

if not ZIP_PATH.exists() or not zipfile.is_zipfile(ZIP_PATH):
    if not FILE_ID:
        raise FileNotFoundError(
            f"Place a valid {ZIP_PATH} in the working directory or set "
            "COURSE_DATA_DRIVE_FILE_ID to the ID of a link-shared dataset."
        )

    downloaded_path = gdown.download(
        id=FILE_ID,
        output=str(ZIP_PATH),
        quiet=False,
    )
    if downloaded_path is None:
        raise RuntimeError(
            "The Google Drive download failed. Check that anyone with the link "
            "can download the configured file."
        )

assert ZIP_PATH.exists(), f"Downloaded file not found: {ZIP_PATH}"
assert zipfile.is_zipfile(ZIP_PATH), (
    "The downloaded file is not a valid ZIP. Check the Google Drive "
    "sharing permission and FILE_ID."
)

print(
    f"Dataset ZIP: {ZIP_PATH} "
    f"({ZIP_PATH.stat().st_size / 1_000_000:.2f} MB)"
)

# Extract the dataset into a clean input directory.
INPUT_DIR = Path("fixed_tolerance_input")

if INPUT_DIR.exists():
    shutil.rmtree(INPUT_DIR)

INPUT_DIR.mkdir(parents=True)
shutil.unpack_archive(str(ZIP_PATH), INPUT_DIR)

# Locate the required files even if the ZIP contains a parent folder.
found = {
    path.name: path
    for path in INPUT_DIR.rglob("*")
    if path.is_file()
}

required = {
    "course_catalog.csv",
    "student_reports.csv",
    "student_ground_truth.json",
    "bundle_queries_with_truth.csv",
    "generation_metadata.json",
}

missing = required - set(found)

assert not missing, (
    f"Missing required files: {sorted(missing)}. "
    f"Files found: {sorted(found)}"
)

# Load the dataset.
catalog = pd.read_csv(found["course_catalog.csv"])
reports_df = pd.read_csv(found["student_reports.csv"])
queries = pd.read_csv(found["bundle_queries_with_truth.csv"])

metadata = json.loads(
    found["generation_metadata.json"].read_text(encoding="utf-8")
)

ground_truth = json.loads(
    found["student_ground_truth.json"].read_text(encoding="utf-8")
)

reports = (
    reports_df
    .set_index("student_id")["language_report"]
    .to_dict()
)

COURSE_IDS = sorted(
    catalog["course_id"].astype(int).tolist()
)

MAX_BUNDLE_SIZE = int(
    metadata["maximum_bundle_size"]
)

TRUE_COMMON_TOLERANCE = float(
    metadata["common_tolerance"]
)

# Convert serialized bundle lists back into Python lists.
for column in ["bundle_a", "bundle_b"]:
    queries[column] = queries[column].apply(json.loads)

# Student splits are generated later inside the mechanism-specific experiment.
student_ids = np.array(sorted(reports_df["student_id"].astype(int).unique()))
print("Students:", len(student_ids))
print("Queries:", len(queries))
print("True tolerance (used only to recover ordinal ground-truth tiers):", TRUE_COMMON_TOLERANCE)

display(catalog)
display(reports_df.sample(3, random_state=SEED))
display(queries.groupby("true_label").size().rename("count"))


## 3. Load the open-source LLM

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

assert torch.cuda.is_available(), "Select a GPU runtime in Colab."

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
).eval()

MODEL_REVISION = getattr(model.config, "_commit_hash", None)
print("Model:", MODEL_ID)
print("Revision:", MODEL_REVISION)
print("GPU:", torch.cuda.get_device_name(0))

## 4. Structured profile prompt and robust validation

In [ ]:
PUBLIC_CATALOG = catalog[
    ["course_id", "course_name", "field", "time_slot", "workload", "description"]
].to_dict("records")

PROFILE_PROMPT = '''
You extract a structured course-preference profile from a student's report.
Use only the report and the public course catalog. Never invent preferences.

For every course with clear evidence, assign exactly one attitude from:
STRONGLY_DISLIKE, DISLIKE, NEUTRAL, LIKE, STRONGLY_LIKE.
Omit courses without clear evidence; omitted courses are treated as NEUTRAL.

Extract complements only when two courses are especially valuable together.
Extract substitutes only when the student indicates overlap, redundancy, or a
desire not to take both. Represent a pair as "smaller_id|larger_id".

Return exactly one JSON object:
{
  "course_attitudes": [{"course_id": 3, "attitude": "LIKE"}],
  "complements": ["0|6"],
  "substitutes": ["2|7"]
}
Do not return Markdown, explanations, confidence scores, tolerance estimates,
or additional keys.
'''.strip()

ATTITUDE_VALUES = {
    "STRONGLY_DISLIKE": -2.0,
    "DISLIKE": -1.0,
    "NEUTRAL": 0.0,
    "LIKE": 1.0,
    "STRONGLY_LIKE": 2.0,
}
ATTITUDE_ALIASES = {
    "STRONG_DISLIKE": "STRONGLY_DISLIKE",
    "VERY_DISLIKE": "STRONGLY_DISLIKE",
    "STRONG_LIKE": "STRONGLY_LIKE",
    "VERY_LIKE": "STRONGLY_LIKE",
}


def extract_json_object(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    start = text.find("{")
    end = text.rfind("}")
    if start < 0 or end < start:
        raise ValueError("No JSON object found.")
    return json.loads(text[start:end + 1])


def canonical_pair(value):
    if isinstance(value, str):
        pieces = value.split("|")
    elif isinstance(value, (list, tuple)) and len(value) == 2:
        pieces = value
    elif isinstance(value, dict):
        pieces = [value.get("course_id_1"), value.get("course_id_2")]
    else:
        raise ValueError(f"Invalid pair: {value}")
    a, b = sorted(map(int, pieces))
    if a == b or a not in COURSE_IDS or b not in COURSE_IDS:
        raise ValueError(f"Invalid course pair: {value}")
    return (a, b)


def validate_profile(parsed):
    if not isinstance(parsed, dict):
        raise ValueError("The parsed response is not an object.")

    course_scores = {course_id: 0.0 for course_id in COURSE_IDS}
    explicitly_scored = set()
    for item in parsed.get("course_attitudes", []):
        course_id = int(item["course_id"])
        attitude = str(item["attitude"]).strip().upper().replace(" ", "_")
        attitude = ATTITUDE_ALIASES.get(attitude, attitude)
        if course_id not in COURSE_IDS:
            raise ValueError(f"Unknown course ID: {course_id}")
        if attitude not in ATTITUDE_VALUES:
            raise ValueError(f"Unknown attitude: {attitude}")
        course_scores[course_id] = ATTITUDE_VALUES[attitude]
        explicitly_scored.add(course_id)

    complements = sorted(set(canonical_pair(x) for x in parsed.get("complements", [])))
    substitutes = sorted(set(canonical_pair(x) for x in parsed.get("substitutes", [])))
    overlap = set(complements) & set(substitutes)
    if overlap:
        # Ambiguous relationships are conservatively discarded.
        complements = [pair for pair in complements if pair not in overlap]
        substitutes = [pair for pair in substitutes if pair not in overlap]

    return {
        "course_scores": course_scores,
        "explicitly_scored_courses": sorted(explicitly_scored),
        "complements": complements,
        "substitutes": substitutes,
    }


@torch.inference_mode()
def generate_profile(report, max_new_tokens=420):
    payload = {"course_catalog": PUBLIC_CATALOG, "student_report": report}
    messages = [
        {"role": "system", "content": PROFILE_PROMPT},
        {"role": "user", "content": json.dumps(payload, ensure_ascii=False)},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated = output[0, inputs.input_ids.shape[1]:]
    raw = tokenizer.decode(generated, skip_special_tokens=True).strip()
    parsed = extract_json_object(raw)
    return validate_profile(parsed), raw

## 5. Smoke test

In [ ]:
smoke_student_id = int(reports_df.student_id.iloc[0])
smoke_profile, smoke_raw = generate_profile(reports[smoke_student_id])
print("Raw output:")
print(smoke_raw)
print("\nValidated profile:")
print(smoke_profile)

## 6. Extract one profile per student with checkpointing

In [ ]:
OUTPUT_DIR = Path("fixed_tolerance_experiment_results")
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT = OUTPUT_DIR / "profile_extractions.jsonl"

completed = {}
if CHECKPOINT.exists():
    for line in CHECKPOINT.read_text(encoding="utf-8").splitlines():
        if line.strip():
            item = json.loads(line)
            completed[int(item["student_id"])] = item
print("Loaded checkpoint rows:", len(completed))

for student_id in tqdm(sorted(map(int, reports)), desc="Extracting profiles"):
    if student_id in completed and completed[student_id].get("status") == "ok":
        continue
    started = time.perf_counter()
    try:
        profile, raw = generate_profile(reports[student_id])
        item = {
            "student_id": student_id,
            "status": "ok",
            "profile": profile,
            "raw_output": raw,
            "error": None,
            "seconds": time.perf_counter() - started,
        }
    except Exception as exc:
        item = {
            "student_id": student_id,
            "status": "error",
            "profile": None,
            "raw_output": None,
            "error": repr(exc),
            "seconds": time.perf_counter() - started,
        }
    completed[student_id] = item
    with CHECKPOINT.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(item) + "\n")

status_df = pd.DataFrame([
    {key: value for key, value in item.items() if key not in {"profile", "raw_output"}}
    for item in completed.values()
]).sort_values("student_id")
display(status_df)
print(status_df.status.value_counts())

failed = status_df.loc[status_df.status != "ok", "student_id"].tolist()
assert not failed, f"Profile extraction failed for students: {failed}. Re-run this cell to retry them."

## 7. Convert profiles and bundle pairs into model features

In [ ]:
profiles = {student_id: item["profile"] for student_id, item in completed.items()}


def bundle_features(profile, bundle):
    bundle = tuple(sorted(map(int, bundle)))
    pairs = set(combinations(bundle, 2))
    score_sum = sum(float(profile["course_scores"].get(str(j), profile["course_scores"].get(j, 0.0))) for j in bundle)
    complement_count = sum(tuple(pair) in pairs for pair in profile["complements"])
    substitute_count = sum(tuple(pair) in pairs for pair in profile["substitutes"])
    explicit_count = sum(j in set(profile["explicitly_scored_courses"]) for j in bundle)
    return np.array([
        score_sum / MAX_BUNDLE_SIZE,
        complement_count / MAX_BUNDLE_SIZE,
        substitute_count / MAX_BUNDLE_SIZE,
        explicit_count / MAX_BUNDLE_SIZE,
    ], dtype=float)


feature_rows = []
for row in queries.itertuples(index=False):
    profile = profiles[int(row.student_id)]
    feature_difference = bundle_features(profile, row.bundle_a) - bundle_features(profile, row.bundle_b)
    feature_rows.append(feature_difference)

X = np.vstack(feature_rows)
FEATURE_NAMES = [
    "course_score_difference",
    "complement_difference",
    "substitute_difference",
    "explicit_coverage_difference",
]
for index, name in enumerate(FEATURE_NAMES):
    queries[name] = X[:, index]

display(queries.head())

The profile-extraction checkpoint uses the same filename as the earlier notebooks. If it is present in the Colab workspace, completed LLM profiles are reused. The allocation experiment below is CPU-only.


## 8. Locked experimental design and mechanism helpers


In [ ]:
N_REPEATED_SPLITS = 20
N_TRAIN_STUDENTS = 64
N_VALIDATION_STUDENTS = 32
N_TEST_STUDENTS = 32
N_VALIDATION_ORDERS = 20
N_TEST_ORDERS = 20
MARKET_CAPACITY_PER_COURSE = 5
LOGIT_C_GRID = [0.01, 0.1, 1.0, 10.0, 100.0]

# Thresholds are measured in within-student standardized predicted-score units.
# tau=0 is the strict-proxy baseline.
TAU_GRID = [0.0, 0.05, 0.10, 0.20, 0.30, 0.50, 0.75, 1.00]
ROBUST_CVAR_QUANTILE = 0.90
ROBUST_CVAR_WEIGHT = 0.50

market_capacities = {int(c): MARKET_CAPACITY_PER_COURSE for c in COURSE_IDS}


def strict_pair_target(rows):
    """Strict ground truth from raw utility; bundle tuple breaks exact equality."""
    targets = []
    for row in rows.itertuples(index=False):
        utility_a = float(row.true_utility_a)
        utility_b = float(row.true_utility_b)
        bundle_a = tuple(sorted(map(int, row.bundle_a)))
        bundle_b = tuple(sorted(map(int, row.bundle_b)))
        targets.append(int(
            utility_a > utility_b
            or (utility_a == utility_b and bundle_a < bundle_b)
        ))
    return np.asarray(targets, dtype=int)


def fit_strict_pairwise_model(rows, return_cv=False):
    """Fit a coherent linear bundle score using strict pair directions only."""
    X_local = rows[FEATURE_NAMES].to_numpy()
    y_local = strict_pair_target(rows)
    groups = rows.student_id.to_numpy()
    cv = GroupKFold(n_splits=min(N_FOLDS, rows.student_id.nunique()))
    c_rows = []
    for C in LOGIT_C_GRID:
        oof = np.full(len(rows), np.nan)
        for fit_idx, hold_idx in cv.split(X_local, y_local, groups):
            model = LogisticRegression(
                C=C, fit_intercept=False, solver="liblinear",
                random_state=SEED, max_iter=2_000,
            )
            model.fit(X_local[fit_idx], y_local[fit_idx])
            oof[hold_idx] = model.predict_proba(X_local[hold_idx])[:, 1]
        predicted = (oof >= 0.5).astype(int)
        c_rows.append({
            "C": C,
            "oof_direction_accuracy": accuracy_score(y_local, predicted),
            "oof_log_loss": float(-np.mean(
                y_local * np.log(np.clip(oof, 1e-12, 1))
                + (1 - y_local) * np.log(np.clip(1 - oof, 1e-12, 1))
            )),
        })
    cv_results = pd.DataFrame(c_rows).sort_values(
        ["oof_direction_accuracy", "oof_log_loss", "C"],
        ascending=[False, True, True],
    ).reset_index(drop=True)
    best_C = float(cv_results.loc[0, "C"])
    model = LogisticRegression(
        C=best_C, fit_intercept=False, solver="liblinear",
        random_state=SEED, max_iter=2_000,
    ).fit(X_local, y_local)
    return (model, best_C, cv_results) if return_cv else model


def heldout_direction_metrics(rows, model):
    y_local = strict_pair_target(rows)
    probabilities = model.predict_proba(rows[FEATURE_NAMES].to_numpy())[:, 1]
    predictions = (probabilities >= 0.5).astype(int)
    return {
        "direction_accuracy": float(accuracy_score(y_local, predictions)),
        "direction_log_loss": float(-np.mean(
            y_local * np.log(np.clip(probabilities, 1e-12, 1))
            + (1 - y_local) * np.log(np.clip(1 - probabilities, 1e-12, 1))
        )),
        "n_pairwise_queries": int(len(rows)),
    }


def build_candidate_market(rows, model):
    """Build standardized proxy scores and hidden true values on one market."""
    predicted, truth = {}, {}
    for student_id, student_rows in rows.groupby("student_id"):
        i = int(student_id)
        bundles = {
            tuple(sorted(map(int, b))) for b in student_rows.bundle_a
        } | {
            tuple(sorted(map(int, b))) for b in student_rows.bundle_b
        }
        raw = {
            b: float(np.dot(model.coef_[0], bundle_features(profiles[i], b)))
            for b in bundles
        }
        raw_values = np.array(list(raw.values()), dtype=float)
        scale = float(np.std(raw_values))
        if scale < 1e-8:
            scale = 1.0
        center = float(np.mean(raw_values))
        predicted[i] = {b: (value - center) / scale for b, value in raw.items()}

        observations = {b: [] for b in bundles}
        for row in student_rows.itertuples(index=False):
            a = tuple(sorted(map(int, row.bundle_a)))
            b = tuple(sorted(map(int, row.bundle_b)))
            observations[a].append(float(row.true_utility_a))
            observations[b].append(float(row.true_utility_b))
        truth[i] = {b: float(np.mean(values)) for b, values in observations.items()}

        # Outside option is always a separate, strictly worst tier.
        predicted[i][tuple()] = min(predicted[i].values()) - 10.0
        truth[i][tuple()] = min(truth[i].values()) - 2.0
    return predicted, truth


def strict_ranks(score_map):
    ordered = sorted(score_map, key=lambda b: (-score_map[b], b))
    return {bundle: rank for rank, bundle in enumerate(ordered)}


def rank_maps(score_maps):
    return {i: strict_ranks(scores) for i, scores in score_maps.items()}


def tier_map(score_map, tau):
    """Create transitive score-anchored tiers; the outside option stays worst."""
    outside = tuple()
    ordered = sorted((b for b in score_map if b), key=lambda b: (-score_map[b], b))
    tiers = {}
    if ordered:
        current_tier = 0
        anchor = score_map[ordered[0]]
        tiers[ordered[0]] = current_tier
        for bundle in ordered[1:]:
            if anchor - score_map[bundle] > tau + 1e-12:
                current_tier += 1
                anchor = score_map[bundle]
            tiers[bundle] = current_tier
        tiers[outside] = current_tier + 1
    else:
        tiers[outside] = 0
    return tiers


def tier_maps(score_maps, tau):
    return {i: tier_map(scores, tau) for i, scores in score_maps.items()}


def predicted_tie_rate(tiers):
    tied = total = 0
    for i in tiers:
        bundles = [b for b in tiers[i] if b]
        for a, b in combinations(bundles, 2):
            total += 1
            tied += int(tiers[i][a] == tiers[i][b])
    return float(tied / total) if total else 0.0


def fits(bundle, remaining):
    return all(remaining[c] > 0 for c in bundle)


def run_weak_rsd(decision_tiers, capacities, seed):
    """Weak RSD with a fixed report-independent lexicographic rule within a tier."""
    rng = np.random.default_rng(seed)
    order = list(map(int, rng.permutation(sorted(decision_tiers))))
    priority_position = {i: position for position, i in enumerate(order)}
    remaining = capacities.copy()
    allocation = {}
    for i in order:
        feasible = [b for b in decision_tiers[i] if fits(b, remaining)]
        best_tier = min(decision_tiers[i][b] for b in feasible)
        chosen = min(b for b in feasible if decision_tiers[i][b] == best_tier)
        allocation[i] = chosen
        for c in chosen:
            remaining[c] -= 1
    return allocation, priority_position


def proxy_pareto_completion(initial_allocation, proxy_tiers, capacities):
    """Proxy-Pareto completion: no proxy loss, maximum total tier improvement."""
    students = sorted(proxy_tiers)
    allowed = {
        i: [
            b for b in proxy_tiers[i]
            if proxy_tiers[i][b] <= proxy_tiers[i][initial_allocation[i]]
        ]
        for i in students
    }
    variables = [(i, b) for i in students for b in allowed[i]]
    index = {key: k for k, key in enumerate(variables)}
    rows, lower, upper = [], [], []

    for i in students:
        rows.append({index[(i, b)]: 1.0 for b in allowed[i]})
        lower.append(1.0); upper.append(1.0)
    for course in COURSE_IDS:
        rows.append({index[(i, b)]: 1.0 for i, b in variables if course in b})
        lower.append(-np.inf); upper.append(float(capacities[course]))

    matrix = lil_matrix((len(rows), len(variables)), dtype=float)
    for r, entries in enumerate(rows):
        for col, value in entries.items():
            matrix[r, col] = value

    tier_gain = np.array([
        proxy_tiers[i][initial_allocation[i]] - proxy_tiers[i][b]
        for i, b in variables
    ], dtype=float)
    # The tiny deterministic term resolves equal-gain optima without changing
    # the integer-valued primary objective.
    deterministic = np.array([
        (student_pos + 1) * 1e-7 + (bundle_pos + 1) * 1e-10
        for student_pos, i in enumerate(students)
        for bundle_pos, b in enumerate(allowed[i])
    ])
    objective = -tier_gain + deterministic
    result = milp(
        c=objective,
        integrality=np.ones(len(variables)),
        bounds=Bounds(np.zeros(len(variables)), np.ones(len(variables))),
        constraints=LinearConstraint(matrix.tocsr(), np.array(lower), np.array(upper)),
        options={"time_limit": 30.0},
    )
    if not result.success:
        raise RuntimeError(f"Pareto-completion MILP failed: {result.message}")
    completed = {i: b for (i, b), x in zip(variables, result.x) if x > 0.5}
    assert len(completed) == len(students)
    assert all(
        proxy_tiers[i][completed[i]] <= proxy_tiers[i][initial_allocation[i]]
        for i in students
    )
    return completed, int(round(tier_gain @ (result.x > 0.5))), result.message


def run_weak_rsd_with_completion(proxy_tiers, capacities, seed):
    initial, priority = run_weak_rsd(proxy_tiers, capacities, seed)
    completed, total_tier_gain, status = proxy_pareto_completion(
        initial, proxy_tiers, capacities
    )
    return completed, priority, initial, total_tier_gain, status


def run_strict_rsd(decision_ranks, capacities, seed):
    tiers = {i: dict(ranks) for i, ranks in decision_ranks.items()}
    return run_weak_rsd(tiers, capacities, seed)


def max_pareto_improvements(allocation, true_ranks, capacities):
    """Maximum strictly improved students subject to nobody becoming worse."""
    students = sorted(true_ranks)
    allowed = {
        i: [b for b in true_ranks[i] if true_ranks[i][b] <= true_ranks[i][allocation[i]]]
        for i in students
    }
    variables = [(i, b) for i in students for b in allowed[i]]
    index = {key: k for k, key in enumerate(variables)}
    rows, lower, upper = [], [], []
    for i in students:
        rows.append({index[(i, b)]: 1.0 for b in allowed[i]})
        lower.append(1.0); upper.append(1.0)
    for course in COURSE_IDS:
        rows.append({index[(i, b)]: 1.0 for i, b in variables if course in b})
        lower.append(-np.inf); upper.append(float(capacities[course]))
    matrix = lil_matrix((len(rows), len(variables)), dtype=float)
    for r, entries in enumerate(rows):
        for col, value in entries.items():
            matrix[r, col] = value
    objective = np.array([
        -float(true_ranks[i][b] < true_ranks[i][allocation[i]])
        for i, b in variables
    ])
    result = milp(
        c=objective,
        integrality=np.ones(len(variables)),
        bounds=Bounds(np.zeros(len(variables)), np.ones(len(variables))),
        constraints=LinearConstraint(matrix.tocsr(), np.array(lower), np.array(upper)),
        options={"time_limit": 30.0},
    )
    if not result.success:
        return np.nan, result.message
    chosen = {i: b for (i, b), x in zip(variables, result.x) if x > 0.5}
    count = sum(true_ranks[i][chosen[i]] < true_ranks[i][allocation[i]] for i in students)
    return int(count), result.message


def priority_blocking_metrics(allocation, true_ranks, capacities, priority_position):
    """Count strictly true-preferred bundles obtainable from empty/lower-priority seats."""
    holders = {c: [] for c in COURSE_IDS}
    for j, bundle in allocation.items():
        for c in bundle:
            holders[c].append(j)
    blocking = []
    for i in sorted(allocation):
        own = allocation[i]
        own_rank = true_ranks[i][own]
        for bundle, rank in true_ranks[i].items():
            if rank >= own_rank:
                continue
            obtainable = True
            for c in bundle:
                if c in own:
                    continue
                course_holders = holders[c]
                if len(course_holders) < capacities[c]:
                    continue
                if any(priority_position[i] < priority_position[j] for j in course_holders):
                    continue
                obtainable = False
                break
            if obtainable:
                blocking.append((i, bundle))
    denominator = sum(max(len(true_ranks[i]) - 1, 1) for i in true_ranks)
    return {
        "priority_blocking_bundles": len(blocking),
        "normalized_blocking_rate": float(len(blocking) / denominator),
        "priority_blocking_students": len({i for i, _ in blocking}),
        "zero_priority_blocking": int(len(blocking) == 0),
    }


def evaluate_allocation(allocation, true_ranks, capacities, priority_position):
    pe_count, solver_status = max_pareto_improvements(allocation, true_ranks, capacities)
    assigned_ranks = np.array([true_ranks[i][allocation[i]] for i in sorted(allocation)])
    metrics = {
        "pareto_improvable_students": pe_count,
        "pareto_violation_rate": float(pe_count / len(allocation)) if np.isfinite(pe_count) else np.nan,
        "pareto_efficient": int(pe_count == 0) if np.isfinite(pe_count) else np.nan,
        "pe_score": 1.0 - pe_count / len(allocation) if np.isfinite(pe_count) else np.nan,
        "mean_true_rank": float(assigned_ranks.mean()),
        "worst_true_rank": int(assigned_ranks.max()),
        "pe_solver_status": solver_status,
    }
    metrics.update(priority_blocking_metrics(
        allocation, true_ranks, capacities, priority_position
    ))
    return metrics


def compare_to_oracle(allocation, oracle_allocation, true_ranks):
    students = sorted(true_ranks)
    losses = np.array([
        true_ranks[i][allocation[i]] - true_ranks[i][oracle_allocation[i]]
        for i in students
    ])
    return {
        "allocation_agreement_rate": float(np.mean([
            allocation[i] == oracle_allocation[i] for i in students
        ])),
        "mean_true_rank_loss_vs_oracle": float(losses.mean()),
        "median_true_rank_loss_vs_oracle": float(np.median(losses)),
        "share_worse_than_oracle": float(np.mean(losses > 0)),
        "share_better_than_oracle": float(np.mean(losses < 0)),
        "worst_student_rank_loss": int(losses.max()),
    }


def cvar(values, quantile=ROBUST_CVAR_QUANTILE):
    values = np.asarray(values, dtype=float)
    cutoff = np.quantile(values, quantile)
    tail = values[values >= cutoff - 1e-12]
    return float(tail.mean())


def robust_risk(values):
    values = np.asarray(values, dtype=float)
    return float(values.mean() + ROBUST_CVAR_WEIGHT * cvar(values))


## 9. Fake-persona attack generator


In [ ]:
# The train/validation/test split seed and these PE thresholds are locked from
# the completed Weak_RSD_Pareto_Completion experiment. They are never retuned
# using attack outcomes.
PE_TAU_BY_SPLIT = {
    0: 1.00, 1: 0.50, 2: 0.75, 3: 1.00, 4: 0.75,
    5: 1.00, 6: 0.50, 7: 1.00, 8: 1.00, 9: 0.30,
    10: 0.50, 11: 0.50, 12: 0.50, 13: 1.00, 14: 0.50,
    15: 1.00, 16: 1.00, 17: 1.00, 18: 1.00, 19: 0.50,
}

N_ATTACK_SPLITS = 10
N_FAKE_REPORTS_PER_STUDENT = 100
N_ATTACK_ORDERS = 20
BEST_K_GRID = [1, 5, 10, 20, 50, 100]

FAKE_STRATEGIES = [
    "random_plausible",
    "field_pivot",
    "top_course_exaggeration",
    "preference_reversal",
    "workload_claim",
    "time_claim",
    "complement_claim",
    "substitute_claim",
    "selective_omission",
    "mimic_peer",
]
assert len(FAKE_STRATEGIES) * 10 == N_FAKE_REPORTS_PER_STUDENT

course_info = catalog.set_index("course_id").to_dict("index")
course_name = {int(c): str(course_info[c]["course_name"]) for c in COURSE_IDS}
field_to_courses = {
    str(field): sorted(map(int, group.course_id))
    for field, group in catalog.groupby("field")
}
time_to_courses = {
    str(slot): sorted(map(int, group.course_id))
    for slot, group in catalog.groupby("time_slot")
}
workload_to_courses = {
    str(level): sorted(map(int, group.course_id))
    for level, group in catalog.groupby("workload")
}

VALUE_TO_ATTITUDE = {
    -2.0: "STRONGLY_DISLIKE", -1.0: "DISLIKE", 0.0: "NEUTRAL",
    1.0: "LIKE", 2.0: "STRONGLY_LIKE",
}


def profile_to_parsed(profile):
    attitudes = []
    explicit = set(map(int, profile.get("explicitly_scored_courses", [])))
    for c in sorted(explicit):
        value = float(profile["course_scores"].get(c, profile["course_scores"].get(str(c), 0.0)))
        value = float(np.clip(round(value), -2, 2))
        attitudes.append({"course_id": c, "attitude": VALUE_TO_ATTITUDE[value]})
    return {
        "course_attitudes": attitudes,
        "complements": [f"{a}|{b}" for a, b in profile.get("complements", [])],
        "substitutes": [f"{a}|{b}" for a, b in profile.get("substitutes", [])],
    }


def parsed_from_scores(scores, complements=None, substitutes=None):
    return {
        "course_attitudes": [
            {"course_id": int(c), "attitude": VALUE_TO_ATTITUDE[float(v)]}
            for c, v in sorted(scores.items()) if float(v) != 0.0
        ],
        "complements": [f"{min(a,b)}|{max(a,b)}" for a, b in (complements or [])],
        "substitutes": [f"{min(a,b)}|{max(a,b)}" for a, b in (substitutes or [])],
    }


def render_fake_report(parsed, strategy, variant, rng):
    attitude_phrases = {
        "STRONGLY_LIKE": ["I am especially eager to take", "My strongest preference is for", "I would prioritize"],
        "LIKE": ["I would like to take", "I am interested in", "I have a positive preference for"],
        "DISLIKE": ["I would rather avoid", "I am not very interested in", "I have a negative preference for"],
        "STRONGLY_DISLIKE": ["I strongly want to avoid", "I would place at the bottom", "I have a strong aversion to"],
        "NEUTRAL": ["I feel neutral about", "I have no strong view about"],
    }
    intros = [
        "Here is how I would describe my course preferences.",
        "My priorities for the coming term are as follows.",
        "When choosing my schedule, these are the preferences I would report.",
        "I have thought about the available courses and would describe my interests this way.",
    ]
    sentences = [rng.choice(intros)]
    items = list(parsed.get("course_attitudes", []))
    rng.shuffle(items)
    for item in items:
        c = int(item["course_id"])
        attitude = str(item["attitude"])
        phrase = rng.choice(attitude_phrases[attitude])
        sentences.append(f"{phrase} {course_name[c]} (course {c}).")
    for value in parsed.get("complements", []):
        a, b = map(int, value.split("|"))
        sentences.append(
            f"{course_name[a]} (course {a}) and {course_name[b]} (course {b}) would be especially valuable together."
        )
    for value in parsed.get("substitutes", []):
        a, b = map(int, value.split("|"))
        sentences.append(
            f"{course_name[a]} (course {a}) and {course_name[b]} (course {b}) overlap for me, so I would prefer not to take both."
        )
    if strategy == "selective_omission":
        sentences.append("The courses not mentioned above are not important to my decision.")
    elif strategy == "random_plausible":
        sentences.append("I am flexible about the rest of the catalog.")
    elif strategy in {"workload_claim", "time_claim"}:
        sentences.append("This scheduling consideration is especially important to me.")
    return " ".join(sentences)


def fake_persona(target_id, strategy, variant, rng, peer_pool):
    truthful = profiles[int(target_id)]
    truth_scores = {
        int(c): float(truthful["course_scores"].get(c, truthful["course_scores"].get(str(c), 0.0)))
        for c in COURSE_IDS
    }
    positive = [c for c in COURSE_IDS if truth_scores[c] > 0]
    negative = [c for c in COURSE_IDS if truth_scores[c] < 0]
    if not positive:
        positive = list(rng.choice(COURSE_IDS, size=min(3, len(COURSE_IDS)), replace=False))
    if not negative:
        negative = [c for c in COURSE_IDS if c not in positive][:3]

    scores, complements, substitutes = {}, [], []
    if strategy == "random_plausible":
        chosen = list(rng.choice(COURSE_IDS, size=min(6, len(COURSE_IDS)), replace=False))
        for c, value in zip(chosen, [2, 1, 1, -1, -1, -2]):
            scores[int(c)] = float(value)
        if variant % 2 == 0:
            complements = [tuple(sorted(chosen[:2]))]

    elif strategy == "field_pivot":
        fields = sorted(field_to_courses)
        field = fields[variant % len(fields)]
        liked = field_to_courses[field][:5]
        scores.update({c: 2.0 if j < 2 else 1.0 for j, c in enumerate(liked)})
        outside = [c for c in COURSE_IDS if c not in liked]
        for c in outside[:2]: scores[c] = -1.0

    elif strategy == "top_course_exaggeration":
        ranked = sorted(COURSE_IDS, key=lambda c: (-truth_scores[c], c))
        for c in ranked[:min(4 + variant % 3, len(ranked))]: scores[c] = 2.0
        for c in ranked[-2:]: scores[c] = -2.0

    elif strategy == "preference_reversal":
        explicit = set(map(int, truthful.get("explicitly_scored_courses", [])))
        for c in explicit:
            scores[c] = float(-np.clip(truth_scores[c], -2, 2))
        complements = [tuple(x) for x in truthful.get("substitutes", [])]
        substitutes = [tuple(x) for x in truthful.get("complements", [])]

    elif strategy == "workload_claim":
        levels = sorted(workload_to_courses)
        liked_level = levels[variant % len(levels)]
        liked = workload_to_courses[liked_level]
        scores.update({c: 2.0 for c in liked[:5]})
        other = [c for c in COURSE_IDS if c not in liked]
        scores.update({c: -1.0 for c in other[:3]})

    elif strategy == "time_claim":
        slots = sorted(time_to_courses)
        liked_slot = slots[variant % len(slots)]
        liked = time_to_courses[liked_slot]
        scores.update({c: 2.0 for c in liked[:5]})
        conflicting = [c for c in COURSE_IDS if c not in liked]
        scores.update({c: -2.0 for c in conflicting[:3]})

    elif strategy == "complement_claim":
        chosen = list(rng.choice(COURSE_IDS, size=min(5, len(COURSE_IDS)), replace=False))
        scores.update({c: 1.0 for c in chosen})
        complements = [tuple(sorted(chosen[:2])), tuple(sorted(chosen[2:4]))]

    elif strategy == "substitute_claim":
        chosen = list(rng.choice(COURSE_IDS, size=min(5, len(COURSE_IDS)), replace=False))
        scores.update({c: 1.0 for c in chosen})
        substitutes = [tuple(sorted(chosen[:2])), tuple(sorted(chosen[2:4]))]

    elif strategy == "selective_omission":
        keep_positive = positive[variant % len(positive)]
        keep_negative = negative[variant % len(negative)]
        scores = {int(keep_positive): 2.0, int(keep_negative): -2.0}

    elif strategy == "mimic_peer":
        candidates = [int(i) for i in peer_pool if int(i) != int(target_id)]
        peer_id = candidates[variant % len(candidates)]
        parsed = profile_to_parsed(profiles[peer_id])
        return parsed, peer_id
    else:
        raise ValueError(strategy)

    return parsed_from_scores(scores, complements, substitutes), None


def profile_distance(fake, truthful):
    fake_scores = np.array([
        float(fake["course_scores"].get(c, fake["course_scores"].get(str(c), 0.0)))
        for c in COURSE_IDS
    ])
    truth_scores = np.array([
        float(truthful["course_scores"].get(c, truthful["course_scores"].get(str(c), 0.0)))
        for c in COURSE_IDS
    ])
    fake_rel = set(map(tuple, fake.get("complements", []))) | set(map(tuple, fake.get("substitutes", [])))
    truth_rel = set(map(tuple, truthful.get("complements", []))) | set(map(tuple, truthful.get("substitutes", [])))
    return {
        "profile_score_l1": float(np.mean(np.abs(fake_scores - truth_scores))),
        "profile_score_l2": float(np.sqrt(np.mean((fake_scores - truth_scores) ** 2))),
        "relationship_symmetric_difference": int(len(fake_rel ^ truth_rel)),
        "explicit_course_count": int(len(fake.get("explicitly_scored_courses", []))),
    }


## 10. Extract fake reports with checkpointing


In [ ]:
ATTACK_DIR = Path("llm_rsd_strategyproofness_work")
ATTACK_DIR.mkdir(exist_ok=True)
FAKE_CHECKPOINT = ATTACK_DIR / "fake_report_profile_extractions.jsonl"

fake_completed = {}
if FAKE_CHECKPOINT.exists():
    for line in FAKE_CHECKPOINT.read_text(encoding="utf-8").splitlines():
        if line.strip():
            item = json.loads(line)
            key = (int(item["split_id"]), int(item["target_student_id"]), int(item["deviation_id"]))
            fake_completed[key] = item
print("Loaded fake-report checkpoint rows:", len(fake_completed))

# Recreate exactly the locked splits and choose one target uniformly at random
# from each of the first ten locked-test markets.
all_ids = np.array(sorted(queries.student_id.unique()))
split_rng = np.random.default_rng(SEED + 2_400)
attack_split_records = []
chosen_target_ids = set()
for split_id in range(N_REPEATED_SPLITS):
    shuffled = split_rng.permutation(all_ids)
    train_ids = set(map(int, shuffled[:N_TRAIN_STUDENTS]))
    validation_ids = set(map(int, shuffled[N_TRAIN_STUDENTS:N_TRAIN_STUDENTS + N_VALIDATION_STUDENTS]))
    test_ids = set(map(int, shuffled[-N_TEST_STUDENTS:]))
    if split_id < N_ATTACK_SPLITS:
        target_rng = np.random.default_rng(SEED + 80_000 + split_id)
        eligible_targets = sorted(test_ids - chosen_target_ids)
        assert eligible_targets, "No unused locked-test target remains."
        target_id = int(target_rng.choice(eligible_targets))
        chosen_target_ids.add(target_id)
        attack_split_records.append({
            "split_id": split_id,
            "train_ids": train_ids,
            "validation_ids": validation_ids,
            "test_ids": test_ids,
            "target_student_id": target_id,
            "pe_tau": float(PE_TAU_BY_SPLIT[split_id]),
        })

fake_manifest_rows = []
for record in tqdm(attack_split_records, desc="Generating and extracting fake reports"):
    split_id = record["split_id"]
    target_id = record["target_student_id"]
    peer_pool = sorted(record["test_ids"])
    for strategy_index, strategy in enumerate(FAKE_STRATEGIES):
        for variant in range(10):
            deviation_id = strategy_index * 10 + variant
            key = (split_id, target_id, deviation_id)
            rng = np.random.default_rng(SEED + 1_000_000 * split_id + 10_000 * target_id + deviation_id)
            intended, mimicked_peer = fake_persona(target_id, strategy, variant, rng, peer_pool)
            report = render_fake_report(intended, strategy, variant, rng)

            if key not in fake_completed or fake_completed[key].get("status") != "ok":
                started = time.perf_counter()
                try:
                    extracted, raw = generate_profile(report)
                    item = {
                        "split_id": split_id,
                        "target_student_id": target_id,
                        "deviation_id": deviation_id,
                        "strategy": strategy,
                        "variant": variant,
                        "mimicked_peer_id": mimicked_peer,
                        "fake_report": report,
                        "intended_persona": intended,
                        "extracted_profile": extracted,
                        "raw_output": raw,
                        "status": "ok",
                        "error": None,
                        "seconds": time.perf_counter() - started,
                    }
                except Exception as exc:
                    item = {
                        "split_id": split_id,
                        "target_student_id": target_id,
                        "deviation_id": deviation_id,
                        "strategy": strategy,
                        "variant": variant,
                        "mimicked_peer_id": mimicked_peer,
                        "fake_report": report,
                        "intended_persona": intended,
                        "extracted_profile": None,
                        "raw_output": None,
                        "status": "error",
                        "error": repr(exc),
                        "seconds": time.perf_counter() - started,
                    }
                fake_completed[key] = item
                with FAKE_CHECKPOINT.open("a", encoding="utf-8") as handle:
                    handle.write(json.dumps(item, ensure_ascii=False) + "\n")

            item = fake_completed[key]
            fake_manifest_rows.append({
                "split_id": split_id,
                "target_student_id": target_id,
                "deviation_id": deviation_id,
                "strategy": strategy,
                "variant": variant,
                "mimicked_peer_id": mimicked_peer,
                "status": item["status"],
                "error": item.get("error"),
                "seconds": item.get("seconds"),
                "fake_report": report,
                "intended_persona_json": json.dumps(intended),
                "extracted_profile_json": json.dumps(item.get("extracted_profile")),
            })

fake_manifest = pd.DataFrame(fake_manifest_rows)
display(fake_manifest.groupby(["strategy", "status"]).size().unstack(fill_value=0))
success_by_target = fake_manifest.assign(ok=fake_manifest.status.eq("ok")).groupby(
    ["split_id", "target_student_id"]
).ok.sum()
print(success_by_target)
assert success_by_target.min() >= 95, "Too many fake-report extraction failures; inspect and rerun."


## 11. Strict and weak-proxy manipulation experiment


In [ ]:
def candidate_bundles(student_rows):
    return {
        tuple(sorted(map(int, b))) for b in student_rows.bundle_a
    } | {
        tuple(sorted(map(int, b))) for b in student_rows.bundle_b
    }


def standardized_score_map(student_rows, fitted_model, profile):
    bundles = candidate_bundles(student_rows)
    raw = {
        b: float(np.dot(fitted_model.coef_[0], bundle_features(profile, b)))
        for b in bundles
    }
    values = np.array(list(raw.values()), dtype=float)
    scale = float(np.std(values))
    if scale < 1e-8: scale = 1.0
    center = float(np.mean(values))
    scores = {b: (value - center) / scale for b, value in raw.items()}
    scores[tuple()] = min(scores.values()) - 10.0
    return scores


def pairwise_ranking_disagreement(rank_a, rank_b):
    bundles = sorted(set(rank_a) & set(rank_b))
    disagreements = total = 0
    for x, y in combinations(bundles, 2):
        total += 1
        disagreements += int((rank_a[x] < rank_a[y]) != (rank_b[x] < rank_b[y]))
    return float(disagreements / total) if total else 0.0


def target_tie_rate(tiers):
    bundles = [b for b in tiers if b]
    pairs = list(combinations(bundles, 2))
    return float(np.mean([tiers[a] == tiers[b] for a, b in pairs])) if pairs else 0.0


attack_rows = []
deviation_metadata_rows = []
truthful_baseline_rows = []

for record in tqdm(attack_split_records, desc="Running manipulation audit"):
    split_id = record["split_id"]
    target_id = record["target_student_id"]
    train = queries[queries.student_id.isin(record["train_ids"])].copy()
    test = queries[queries.student_id.isin(record["test_ids"])].copy()
    model, best_C, _ = fit_strict_pairwise_model(train, return_cv=True)

    truthful_scores, true_scores = build_candidate_market(test, model)
    true_ranks = rank_maps(true_scores)
    truthful_strict_ranks = rank_maps(truthful_scores)
    pe_tau = record["pe_tau"]
    truthful_weak_tiers = tier_maps(truthful_scores, pe_tau)
    target_rows = test[test.student_id.eq(target_id)]

    baseline = {"strict": {}, "weak_pe_completion": {}}
    for order_run in range(N_ATTACK_ORDERS):
        order_seed = SEED + 40_000_000 + 100_000 * split_id + order_run
        strict_allocation, strict_priority = run_strict_rsd(
            truthful_strict_ranks, market_capacities, order_seed
        )
        weak_allocation, weak_priority, _, _, _ = run_weak_rsd_with_completion(
            truthful_weak_tiers, market_capacities, order_seed
        )
        assert strict_priority == weak_priority
        baseline["strict"][order_run] = strict_allocation[target_id]
        baseline["weak_pe_completion"][order_run] = weak_allocation[target_id]
        for mechanism, allocation in [
            ("strict", strict_allocation), ("weak_pe_completion", weak_allocation)
        ]:
            bundle = allocation[target_id]
            truthful_baseline_rows.append({
                "split_id": split_id, "target_student_id": target_id,
                "mechanism": mechanism, "order_run": order_run,
                "priority_position": strict_priority[target_id],
                "truthful_bundle": str(bundle),
                "truthful_true_rank": true_ranks[target_id][bundle],
                "truthful_true_utility": true_scores[target_id][bundle],
                "selected_pe_tau": pe_tau,
            })

    target_fake_rows = fake_manifest[
        fake_manifest.split_id.eq(split_id)
        & fake_manifest.target_student_id.eq(target_id)
        & fake_manifest.status.eq("ok")
    ].sort_values("deviation_id")

    for fake_row in target_fake_rows.itertuples(index=False):
        key = (split_id, target_id, int(fake_row.deviation_id))
        fake_profile = fake_completed[key]["extracted_profile"]
        fake_target_scores = standardized_score_map(target_rows, model, fake_profile)
        fake_strict_target_ranks = strict_ranks(fake_target_scores)
        fake_weak_target_tiers = tier_map(fake_target_scores, pe_tau)

        fake_strict_ranks = {i: dict(ranks) for i, ranks in truthful_strict_ranks.items()}
        fake_strict_ranks[target_id] = fake_strict_target_ranks
        fake_weak_tiers = {i: dict(tiers) for i, tiers in truthful_weak_tiers.items()}
        fake_weak_tiers[target_id] = fake_weak_target_tiers

        distance = profile_distance(fake_profile, profiles[target_id])
        metadata = {
            "split_id": split_id, "target_student_id": target_id,
            "deviation_id": int(fake_row.deviation_id),
            "strategy": fake_row.strategy, "variant": int(fake_row.variant),
            "strict_ranking_disagreement": pairwise_ranking_disagreement(
                truthful_strict_ranks[target_id], fake_strict_target_ranks
            ),
            "truthful_target_tie_rate": target_tie_rate(truthful_weak_tiers[target_id]),
            "fake_target_tie_rate": target_tie_rate(fake_weak_target_tiers),
            **distance,
        }
        deviation_metadata_rows.append(metadata)

        for order_run in range(N_ATTACK_ORDERS):
            order_seed = SEED + 40_000_000 + 100_000 * split_id + order_run
            fake_strict_allocation, strict_priority = run_strict_rsd(
                fake_strict_ranks, market_capacities, order_seed
            )
            fake_weak_allocation, weak_priority, _, _, _ = run_weak_rsd_with_completion(
                fake_weak_tiers, market_capacities, order_seed
            )
            assert strict_priority == weak_priority

            for mechanism, allocation in [
                ("strict", fake_strict_allocation),
                ("weak_pe_completion", fake_weak_allocation),
            ]:
                truthful_bundle = baseline[mechanism][order_run]
                fake_bundle = allocation[target_id]
                truthful_rank = true_ranks[target_id][truthful_bundle]
                fake_rank = true_ranks[target_id][fake_bundle]
                truthful_utility = true_scores[target_id][truthful_bundle]
                fake_utility = true_scores[target_id][fake_bundle]
                attack_rows.append({
                    "split_id": split_id, "target_student_id": target_id,
                    "deviation_id": int(fake_row.deviation_id),
                    "strategy": fake_row.strategy, "variant": int(fake_row.variant),
                    "mechanism": mechanism, "order_run": order_run,
                    "priority_position": strict_priority[target_id],
                    "selected_pe_tau": pe_tau,
                    "truthful_bundle": str(truthful_bundle),
                    "fake_bundle": str(fake_bundle),
                    "truthful_true_rank": truthful_rank,
                    "fake_true_rank": fake_rank,
                    "true_rank_gain": int(truthful_rank - fake_rank),
                    "truthful_true_utility": truthful_utility,
                    "fake_true_utility": fake_utility,
                    "true_utility_gain": float(fake_utility - truthful_utility),
                    "profitable": int(fake_rank < truthful_rank),
                    "harmful": int(fake_rank > truthful_rank),
                    "unchanged": int(fake_rank == truthful_rank),
                })

attack_results = pd.DataFrame(attack_rows)
deviation_metadata = pd.DataFrame(deviation_metadata_rows)
truthful_baselines = pd.DataFrame(truthful_baseline_rows)
print("Attack-order comparisons:", len(attack_results))
display(attack_results.groupby("mechanism")[["profitable", "harmful", "unchanged", "true_rank_gain"]].mean())


## 12. Ex-ante and stochastic-dominance tests


In [ ]:
def empirical_sd_comparison(fake_ranks, truthful_ranks):
    fake_ranks = np.asarray(fake_ranks, dtype=int)
    truthful_ranks = np.asarray(truthful_ranks, dtype=int)
    cutoffs = range(0, int(max(fake_ranks.max(), truthful_ranks.max())) + 1)
    fake_cdf = np.array([np.mean(fake_ranks <= q) for q in cutoffs])
    truthful_cdf = np.array([np.mean(truthful_ranks <= q) for q in cutoffs])
    fake_sd_truth = bool(np.all(fake_cdf >= truthful_cdf - 1e-12) and np.any(fake_cdf > truthful_cdf + 1e-12))
    truth_sd_fake = bool(np.all(truthful_cdf >= fake_cdf - 1e-12) and np.any(truthful_cdf > fake_cdf + 1e-12))
    return {
        "fake_sd_dominates_truthful": int(fake_sd_truth),
        "truthful_sd_dominates_fake": int(truth_sd_fake),
        "max_cdf_advantage": float(np.max(fake_cdf - truthful_cdf)),
        "min_cdf_advantage": float(np.min(fake_cdf - truthful_cdf)),
    }


report_summary_rows = []
group_columns = ["split_id", "target_student_id", "deviation_id", "strategy", "variant", "mechanism"]
for keys, group in attack_results.groupby(group_columns):
    row = dict(zip(group_columns, keys))
    row.update({
        "expost_profitable_rate": float(group.profitable.mean()),
        "expost_harmful_rate": float(group.harmful.mean()),
        "expost_unchanged_rate": float(group.unchanged.mean()),
        "expected_true_rank_gain": float(group.true_rank_gain.mean()),
        "median_true_rank_gain": float(group.true_rank_gain.median()),
        "best_order_rank_gain": int(group.true_rank_gain.max()),
        "worst_order_rank_gain": int(group.true_rank_gain.min()),
        "expected_true_utility_gain": float(group.true_utility_gain.mean()),
        "exante_rank_profitable": int(group.true_rank_gain.mean() > 1e-12),
        "exante_utility_profitable": int(group.true_utility_gain.mean() > 1e-12),
    })
    row.update(empirical_sd_comparison(
        group.fake_true_rank.to_numpy(), group.truthful_true_rank.to_numpy()
    ))
    report_summary_rows.append(row)

report_summary = pd.DataFrame(report_summary_rows).merge(
    deviation_metadata,
    on=["split_id", "target_student_id", "deviation_id", "strategy", "variant"],
    how="left",
    validate="many_to_one",
)

target_summary = (
    report_summary.groupby(["split_id", "target_student_id", "mechanism"], as_index=False)
    .agg(
        fake_reports_tested=("deviation_id", "count"),
        exante_profitable_report_rate=("exante_rank_profitable", "mean"),
        sd_profitable_report_rate=("fake_sd_dominates_truthful", "mean"),
        any_exante_profitable=("exante_rank_profitable", "max"),
        any_sd_profitable=("fake_sd_dominates_truthful", "max"),
        best_expected_rank_gain=("expected_true_rank_gain", "max"),
        mean_expected_rank_gain=("expected_true_rank_gain", "mean"),
        worst_expected_rank_gain=("expected_true_rank_gain", "min"),
    )
)

mechanism_summary = (
    report_summary.groupby("mechanism", as_index=False)
    .agg(
        fake_reports=("deviation_id", "count"),
        exante_profitable_report_rate=("exante_rank_profitable", "mean"),
        sd_profitable_report_rate=("fake_sd_dominates_truthful", "mean"),
        mean_expected_rank_gain=("expected_true_rank_gain", "mean"),
        conditional_profitable_rank_gain=("expected_true_rank_gain", lambda x: float(x[x > 0].mean()) if np.any(x > 0) else 0.0),
        mean_expost_profitable_rate=("expost_profitable_rate", "mean"),
        mean_expost_harmful_rate=("expost_harmful_rate", "mean"),
        share_targets_with_profitable_deviation=("target_student_id", lambda x: np.nan),
    )
)
target_rates = target_summary.groupby("mechanism").agg(
    share_targets_with_profitable_deviation=("any_exante_profitable", "mean"),
    share_targets_with_sd_profitable_deviation=("any_sd_profitable", "mean"),
).reset_index()
mechanism_summary = mechanism_summary.drop(columns="share_targets_with_profitable_deviation").merge(
    target_rates, on="mechanism", how="left"
)

strategy_summary = (
    report_summary.groupby(["mechanism", "strategy"], as_index=False)
    .agg(
        reports=("deviation_id", "count"),
        exante_profitable_rate=("exante_rank_profitable", "mean"),
        sd_profitable_rate=("fake_sd_dominates_truthful", "mean"),
        mean_expected_rank_gain=("expected_true_rank_gain", "mean"),
        mean_expost_profitable_rate=("expost_profitable_rate", "mean"),
        mean_ranking_disagreement=("strict_ranking_disagreement", "mean"),
        mean_fake_tie_rate=("fake_target_tie_rate", "mean"),
    )
)

best_k_rows = []
for (split_id, target_id, mechanism), group in report_summary.groupby(
    ["split_id", "target_student_id", "mechanism"]
):
    # A deterministic random permutation prevents the best-of-K curve from
    # being confounded by the arbitrary ordering of strategy families.
    group = group.sort_values("deviation_id")
    search_rng = np.random.default_rng(
        SEED + 9_000_000 + 10_000 * int(split_id) + 10 * int(target_id)
    )
    gains = group.expected_true_rank_gain.to_numpy()[search_rng.permutation(len(group))]
    for k in BEST_K_GRID:
        available = gains[:min(k, len(gains))]
        best_k_rows.append({
            "split_id": split_id, "target_student_id": target_id,
            "mechanism": mechanism, "K": k,
            "best_expected_rank_gain": float(np.max(available)),
            "found_profitable_deviation": int(np.max(available) > 1e-12),
        })
best_k_results = pd.DataFrame(best_k_rows)
best_k_summary = best_k_results.groupby(["mechanism", "K"], as_index=False).agg(
    mean_best_expected_rank_gain=("best_expected_rank_gain", "mean"),
    probability_find_profitable=("found_profitable_deviation", "mean"),
)

display(mechanism_summary)
display(target_summary)
display(strategy_summary)
display(best_k_summary)


## 13. Visualize and export results


In [ ]:
FIGURE_DIR = Path("llm_rsd_strategyproofness_figures")
FIGURE_DIR.mkdir(exist_ok=True)
mechanism_order = ["strict", "weak_pe_completion"]
labels = ["Strict proxy RSD", "Weak PE + completion"]
colors = ["#2563eb", "#7c3aed"]

plot_mechanism = mechanism_summary.set_index("mechanism").reindex(mechanism_order)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].bar(labels, plot_mechanism.exante_profitable_report_rate, color=colors)
axes[0].set(title="Ex-ante profitable reports", ylabel="Share of fake reports", ylim=(0, 1))
axes[1].bar(labels, plot_mechanism.sd_profitable_report_rate, color=colors)
axes[1].set(title="SD-profitable reports", ylabel="Share of fake reports", ylim=(0, 1))
axes[2].bar(labels, plot_mechanism.share_targets_with_profitable_deviation, color=colors)
axes[2].set(title="Targets with at least one profitable report", ylabel="Share of target students", ylim=(0, 1))
for ax in axes:
    ax.grid(axis="y", alpha=0.25)
    ax.tick_params(axis="x", rotation=15)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "mechanism_manipulability.png", dpi=180)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for mechanism, label, color in zip(mechanism_order, labels, colors):
    group = best_k_summary[best_k_summary.mechanism.eq(mechanism)]
    axes[0].plot(group.K, group.mean_best_expected_rank_gain, marker="o", label=label, color=color)
    axes[1].plot(group.K, group.probability_find_profitable, marker="o", label=label, color=color)
axes[0].set(title="Best-of-K search gain", xlabel="Fake reports tried (K)", ylabel="Mean best expected true-rank gain")
axes[1].set(title="Probability of finding a profitable report", xlabel="Fake reports tried (K)", ylabel="Target-student share", ylim=(0, 1.02))
for ax in axes:
    ax.grid(alpha=0.25); ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "best_of_k_search.png", dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(13, 6))
pivot = strategy_summary.pivot(index="strategy", columns="mechanism", values="exante_profitable_rate").reindex(FAKE_STRATEGIES)
x = np.arange(len(pivot))
width = 0.38
ax.bar(x - width/2, pivot["strict"], width, label="Strict proxy RSD", color=colors[0])
ax.bar(x + width/2, pivot["weak_pe_completion"], width, label="Weak PE + completion", color=colors[1])
ax.set(title="Profitable deviations by report strategy", ylabel="Ex-ante profitable report rate", ylim=(0, 1))
ax.set_xticks(x); ax.set_xticklabels(pivot.index, rotation=35, ha="right")
ax.grid(axis="y", alpha=0.25); ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "strategy_success_rates.png", dpi=180)
plt.show()

OUTPUT_DIR = Path("llm_rsd_strategyproofness_results")
OUTPUT_DIR.mkdir(exist_ok=True)
fake_manifest.to_csv(OUTPUT_DIR / "fake_report_manifest.csv", index=False)
deviation_metadata.to_csv(OUTPUT_DIR / "deviation_metadata.csv", index=False)
truthful_baselines.to_csv(OUTPUT_DIR / "truthful_baselines.csv", index=False)
attack_results.to_csv(OUTPUT_DIR / "deviation_order_results.csv", index=False)
report_summary.to_csv(OUTPUT_DIR / "deviation_report_summary.csv", index=False)
target_summary.to_csv(OUTPUT_DIR / "target_student_summary.csv", index=False)
mechanism_summary.to_csv(OUTPUT_DIR / "mechanism_summary.csv", index=False)
strategy_summary.to_csv(OUTPUT_DIR / "strategy_summary.csv", index=False)
best_k_results.to_csv(OUTPUT_DIR / "best_k_target_results.csv", index=False)
best_k_summary.to_csv(OUTPUT_DIR / "best_k_summary.csv", index=False)
for path in FIGURE_DIR.glob("*.png"):
    shutil.copy2(path, OUTPUT_DIR / path.name)

metadata = {
    "seed": SEED,
    "audit_name": "empirical manipulability of LLM-mediated RSD",
    "interpretation_warning": "Deviation rates are conditional on the defined attack generator; one profitable deviation is a counterexample, while failure to find one is not a proof of strategy-proofness.",
    "mechanisms": ["strict LLM-proxy RSD", "PE-calibrated weak LLM-proxy RSD plus proxy-Pareto completion"],
    "n_target_students": N_ATTACK_SPLITS,
    "target_selection": "one distinct uniformly random locked-test student from each of the first ten repeated splits",
    "fake_reports_per_target": N_FAKE_REPORTS_PER_STUDENT,
    "attack_orders": N_ATTACK_ORDERS,
    "common_random_numbers": True,
    "fake_strategy_families": FAKE_STRATEGIES,
    "best_k_grid": BEST_K_GRID,
    "locked_pe_thresholds_by_split": PE_TAU_BY_SPLIT,
    "evaluation_preference": "hidden strict true bundle ranking; hidden cardinal utility is secondary only",
    "domain": "observed candidate bundles plus outside option",
}
(OUTPUT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
archive = Path(shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR))
print("Saved:", archive.resolve())
from google.colab import files
files.download(str(archive))
